# Module 3: Predictive Modeling in R with Tidymodels
**ACS Predictive Analytics Curriculum**

Concepts covered:
- Logistic regression in R (glm + tidymodels)
- Random Forest with ranger
- XGBoost in R
- Tidymodels pipeline (recipe + workflow + tune)
- Group-aware cross-validation by family_id
- Model comparison on AUC-PR


In [ ]:
install.packages(c('tidymodels','ranger','xgboost','themis',
  'yardstick','probably','vip','discrim'), repos='https://cran.rstudio.com/', quiet=TRUE)
library(tidymodels)
library(ranger)
library(xgboost)
library(themis)
library(vip)
cat('Packages loaded\n')

## SECTION 3.1: Load and Prepare Data

In [ ]:
library(tidyverse)
scr      <- read_csv('data/acs_scr_reports.csv', show_col_types=FALSE)
features <- read_csv('data/acs_features.csv',    show_col_types=FALSE)

# Build clean modeling dataset
model_data <- features %>%
  mutate(
    has_prior_report       = prior_reports_12mo > 0,
    days_since_last_report = replace_na(days_since_last_report, -1),
    substance_flag_missing = is.na(substance_use_flag),
    substance_use_flag     = replace_na(substance_use_flag,
                               median(substance_use_flag, na.rm=TRUE)),
    contact_missing        = is.na(days_to_first_contact),
    days_to_first_contact  = replace_na(days_to_first_contact,
                               median(days_to_first_contact, na.rm=TRUE)),
    high_caseload          = caseworker_caseload > 60,
    compounded_risk        = dv_history_flag + shelter_involvement_flag + prior_substantiated_flag,
    # TARGET must be a factor for classification
    needs_investigative_consultation = factor(
      needs_investigative_consultation,
      levels = c(0,1), labels = c('No','Yes')
    )
  )

cat('Model data ready:', nrow(model_data), 'rows\n')
cat('Positive rate:', round(mean(model_data$needs_investigative_consultation=='Yes')*100,1), '%\n')

## SECTION 3.2: Group-Aware Train/Test Split
**Why group-aware?** Same family appears multiple times.
Random split leaks family patterns across train/test.
We split at the FAMILY level — all reports from FAM00103
go entirely into train OR test, never both.

In [ ]:
set.seed(42)

# Get unique families and assign to train/test
unique_families <- model_data %>%
  distinct(family_id) %>%
  mutate(split = sample(c('train','test'), n(),
                        replace=TRUE, prob=c(0.8, 0.2)))

model_data <- model_data %>%
  left_join(unique_families, by='family_id')

train_data <- model_data %>% filter(split == 'train') %>% select(-split)
test_data  <- model_data %>% filter(split == 'test')  %>% select(-split)

cat('Train:', nrow(train_data), 'rows |',
    round(mean(train_data$needs_investigative_consultation=='Yes')*100,1), '% positive\n')
cat('Test: ', nrow(test_data),  'rows |',
    round(mean(test_data$needs_investigative_consultation=='Yes')*100,1),  '% positive\n')
cat('Family overlap:', sum(unique(train_data$family_id) %in% unique(test_data$family_id)), '(must be 0)\n')

## SECTION 3.3: Define the Recipe (Preprocessing Pipeline)

In [ ]:
# The recipe defines ALL preprocessing steps
# Critically: SMOTE is applied INSIDE the recipe
# so it only affects training folds, never test data

FEATURES <- c('prior_reports_12mo','prior_substantiated_flag',
               'days_since_last_report','has_prior_report',
               'n_children_in_household','child_age_under_5',
               'dv_history_flag','substance_use_flag','substance_flag_missing',
               'shelter_involvement_flag','days_to_first_contact','contact_missing',
               'reporter_accuracy_score','caseworker_caseload','high_caseload',
               'compounded_risk')

acs_recipe <- recipe(
    needs_investigative_consultation ~ .,
    data = train_data %>% select(all_of(c(FEATURES, 'needs_investigative_consultation')))
  ) %>%
  # Convert logicals to numeric
  step_mutate(across(where(is.logical), as.integer)) %>%
  # Normalize continuous features (required for logistic regression)
  step_normalize(all_numeric_predictors()) %>%
  # SMOTE: create synthetic minority class samples
  # Applied INSIDE CV fold only - prevents leakage
  step_smote(needs_investigative_consultation, over_ratio=0.8)

# Preview what the recipe does
prep(acs_recipe) %>% bake(new_data=NULL) %>% head(3)

## SECTION 3.4: Define Three Models

In [ ]:
# MODEL 1: Logistic Regression
# Interpretable baseline - coefficients = log-odds
lr_spec <- logistic_reg(penalty=0.01, mixture=1) %>%
  set_engine('glmnet') %>%
  set_mode('classification')

# MODEL 2: Random Forest
# 100 trees, parallel, robust to noise
rf_spec <- rand_forest(trees=100, mtry=4, min_n=5) %>%
  set_engine('ranger', importance='impurity') %>%
  set_mode('classification')

# MODEL 3: XGBoost (Gradient Boosting)
# Sequential trees correcting residuals
xgb_spec <- boost_tree(trees=100, learn_rate=0.1,
                        tree_depth=3, loss_reduction=0.01) %>%
  set_engine('xgboost') %>%
  set_mode('classification')

cat('Three model specs defined\n')

## SECTION 3.5: Build Workflows and Cross-Validate

In [ ]:
# Group K-Fold: keeps all family reports in same fold
# This is the key difference from regular vfold_cv()
folds <- group_vfold_cv(
  train_data %>%
    select(all_of(c(FEATURES, 'needs_investigative_consultation', 'family_id'))),
  group = 'family_id',
  v = 5
)
cat('Created', length(folds$splits), 'group-aware folds\n')

# Metric set: AUC-PR is primary, AUC-ROC secondary
# AUC-PR not distorted by class imbalance unlike AUC-ROC
acs_metrics <- metric_set(average_precision, roc_auc, f_meas)

# Build and evaluate all three models
results <- list()

for(model_name in c('logistic','random_forest','xgboost')) {
  spec <- list(logistic=lr_spec, random_forest=rf_spec, xgboost=xgb_spec)[[model_name]]

  wf <- workflow() %>%
    add_recipe(acs_recipe) %>%
    add_model(spec)

  cv_res <- fit_resamples(
    wf, folds,
    metrics = acs_metrics,
    control = control_resamples(save_pred=TRUE)
  )

  results[[model_name]] <- cv_res
  metrics <- collect_metrics(cv_res)
  cat('\n', model_name, '\n')
  print(metrics %>% select(.metric, mean, std_err) %>% arrange(.metric))
}

## SECTION 3.6: Compare Models and Select Winner

In [ ]:
# Collect all metrics into one comparison table
comparison <- bind_rows(
  collect_metrics(results$logistic)     %>% mutate(model='Logistic Regression'),
  collect_metrics(results$random_forest)%> mutate(model='Random Forest'),
  collect_metrics(results$xgboost)      %>% mutate(model='XGBoost')
) %>%
  filter(.metric %in% c('average_precision','roc_auc')) %>%
  select(model, .metric, mean, std_err) %>%
  pivot_wider(names_from=.metric, values_from=c(mean,std_err)) %>%
  arrange(desc(mean_average_precision))

print(comparison)

# Winner: model with highest AUC-PR
# AUC-PR is our primary metric because:
# - Not distorted by class imbalance (unlike AUC-ROC)
# - Directly measures how well we find the positive class
# - Baseline = positive rate in data (~0.25)
# - Our model must beat this baseline to be useful

## SECTION 3.7: Fit Final Model and Evaluate on Test Set

In [ ]:
# Fit XGBoost (assume it wins - replace with winner from above)
final_wf <- workflow() %>%
  add_recipe(acs_recipe) %>%
  add_model(xgb_spec)

final_fit <- final_wf %>%
  fit(data = train_data %>%
    select(all_of(c(FEATURES, 'needs_investigative_consultation'))))

# Predict on test set
test_preds <- final_fit %>%
  predict(test_data, type='prob') %>%
  bind_cols(test_data %>% select(needs_investigative_consultation, family_id, all_of(FEATURES)))

# Evaluate
test_preds %>%
  roc_auc(truth=needs_investigative_consultation, .pred_Yes)

test_preds %>%
  average_precision(truth=needs_investigative_consultation, .pred_Yes)

## SECTION 3.8: Feature Importance

In [ ]:
# Extract feature importance from final XGBoost model
# vip package makes this easy across model types

final_fit %>%
  extract_fit_parsnip() %>%
  vip(num_features=15) +
  labs(title='Feature Importance: XGBoost Risk Model',
       subtitle='Higher = more important for predicting consultation need') +
  theme_minimal(base_size=12)

# DOMAIN INTERPRETATION:
# prior_reports_12mo at top -> history is strongest predictor
# dv_history_flag high -> domestic violence = key risk signal
# If caseworker_caseload is high -> flag for equity review
# (systemic variable, not true family risk signal)

## EXERCISE 3.1
Tune the XGBoost model using tune_grid():
- Try learn_rate: c(0.05, 0.10, 0.15)
- Try tree_depth: c(3, 4, 5)
- Pick best combination by AUC-PR
- Does tuning improve over the defaults?

## EXERCISE 3.2
Compare feature importance between Random Forest and XGBoost.
Do they agree on the top 5 features?
Where they disagree - which should you trust more and why?
(Hint: permutation importance vs tree impurity)
